# Supply Chain Optimization
**Aggregate Production Planning, Promotion Strategy & Competitive Analysis**

Taha Furkan Torun, Nil Erva Demir, Miray Köse

## Setup

In [1]:
from pyomo.environ import *

In [2]:
initial_inventory = 250
production_cost = 1000
holding_cost = 100
subcontracting_cost = 1250
capacity = 160
fixed_employee_cost = 100 * 10 * 320 *12

## Demand Scenarios

Base demand and modified demand vectors for each promotional scenario. April and August promotions increase demand by 40% in the promotion month and pull forward 20% from each of the following two months.

In [3]:
base_demand = {1: 380, 2: 310, 3: 370, 4: 610, 5: 380, 6: 300, 7: 390, 8: 220, 9: 340, 10: 395, 11: 402, 12: 397}

base_price = {t: 2800 for t in range(1, 13)}

demand_april = base_demand.copy()
price_april = base_price.copy()
price_april[4] = 2520
demand_april[4] = 990
demand_april[5] = 304
demand_april[6] = 240

demand_august = base_demand.copy()
price_august = base_price.copy()
price_august[8] = 2520
demand_august[8] = 455
demand_august[9] = 272
demand_august[10] = 316

## Model Definitions

Two reusable Pyomo LP models built on the same constraint structure — one minimises total cost, the other maximises profit. Decision variables are monthly production `P[t]`, subcontracting `S[t]`, and inventory `I[t]`.

In [4]:
def build_cost_model(demand):
    model = ConcreteModel()
    model.months = RangeSet(1, 12)
    model.P = Var(model.months, domain=NonNegativeReals)
    model.S = Var(model.months, domain=NonNegativeReals)
    model.I = Var(model.months, domain=NonNegativeReals)
    model.D = Param(model.months, initialize=demand)
    
    cost_expr = sum(production_cost * model.P[t] + subcontracting_cost * model.S[t] + holding_cost * model.I[t] for t in model.months) + fixed_employee_cost
    model.total_cost = Objective(expr=cost_expr, sense=minimize)
    
    def inventory_balance_rule(model, t):
        if t == 1:
            return model.I[t] == initial_inventory + model.P[t] + model.S[t] - model.D[t]
        else:
            return model.I[t] == model.I[t-1] + model.P[t] + model.S[t] - model.D[t]
        
    model.inventory_balance = Constraint(model.months, rule=inventory_balance_rule)

    def capacity_rule(model, t):
        return model.P[t] <= capacity
    
    model.capacity_constraint = Constraint(model.months, rule=capacity_rule)

    def safety_stock_rule(model, t):
        if t == 12:
            return Constraint.Skip
        return model.I[t] >= 100
    
    model.safety_stock_constraint = Constraint(model.months, rule=safety_stock_rule)
    model.final_inventory_constraint = Constraint(expr=model.I[12] == 250)

    return model

In [5]:
def build_profit_model(demand, price):
    model = ConcreteModel()
    model.months = RangeSet(1, 12)
    model.P = Var(model.months, domain=NonNegativeReals)
    model.S = Var(model.months, domain=NonNegativeReals)
    model.I = Var(model.months, domain=NonNegativeReals)
    model.D = Param(model.months, initialize=demand)
    model.price = Param(model.months, initialize=price)

    profit_expr = sum(model.price[t] * model.D[t] - production_cost * model.P[t] - subcontracting_cost * model.S[t] - holding_cost * model.I[t] for t in model.months) - fixed_employee_cost

    model.profit = Objective(expr=profit_expr, sense=maximize)

    def inventory_balance_rule(model, t):
        if t == 1:
            return model.I[t] == initial_inventory + model.P[t] + model.S[t] - model.D[t]
        else:
            return model.I[t] == model.I[t-1] + model.P[t] + model.S[t] - model.D[t]
        
    model.inventory_balance = Constraint(model.months, rule=inventory_balance_rule)

    def capacity_rule(model, t):
        return model.P[t] <= capacity
    
    model.capacity_constraint = Constraint(model.months, rule=capacity_rule)

    def safety_stock_rule(model, t):
        if t == 12:
            return Constraint.Skip
        return model.I[t] >= 100
    
    model.safety_stock_constraint = Constraint(model.months, rule=safety_stock_rule)
    model.final_inventory_constraint = Constraint(expr=model.I[12] == 250)

    return model





## Part 1 — Cost Minimisation

Baseline plan: minimise total operational costs over 12 months. Internal production is capped at 160 tons/month. Subcontracting fills demand gaps at $1,250/ton. Inventory held at 100-ton safety stock floor throughout.

In [6]:
model_q1 = build_cost_model(base_demand)
solver = SolverFactory('glpk')
result_q1 = solver.solve(model_q1)

print("Total Cost:", value(model_q1.total_cost))

print("Production Plan:")
for t in model_q1.months:
    print(f"Month {t}: "f"Produce {value(model_q1.P[t])} units, "f"Subcontract {value(model_q1.S[t])} units, "f"Inventory {value(model_q1.I[t])} units")

QUESTION 1
Total Cost: 9112500.0
Production Plan:
Month 1: Produce 160.0 units, Subcontract 70.0 units, Inventory 100.0 units
Month 2: Produce 160.0 units, Subcontract 150.0 units, Inventory 100.0 units
Month 3: Produce 160.0 units, Subcontract 210.0 units, Inventory 100.0 units
Month 4: Produce 160.0 units, Subcontract 450.0 units, Inventory 100.0 units
Month 5: Produce 160.0 units, Subcontract 220.0 units, Inventory 100.0 units
Month 6: Produce 160.0 units, Subcontract 140.0 units, Inventory 100.0 units
Month 7: Produce 160.0 units, Subcontract 230.0 units, Inventory 100.0 units
Month 8: Produce 160.0 units, Subcontract 60.0 units, Inventory 100.0 units
Month 9: Produce 160.0 units, Subcontract 180.0 units, Inventory 100.0 units
Month 10: Produce 160.0 units, Subcontract 235.0 units, Inventory 100.0 units
Month 11: Produce 160.0 units, Subcontract 242.0 units, Inventory 100.0 units
Month 12: Produce 160.0 units, Subcontract 387.0 units, Inventory 250.0 units


## Part 2 — Profit Maximisation

Same model re-solved with profit as objective (revenue at $2,800/ton minus all costs). Since price is fixed and all demand must be met, the production plan is identical to Part 1 — the choice of objective only affects how we interpret the result.

In [7]:
model_q2 = build_profit_model(base_demand, base_price)
solver = SolverFactory('glpk')
result_q2 = solver.solve(model_q2)

print("Optimal Profit:", value(model_q2.profit))

print("Production Plan:")
for t in model_q2.months:
    print(f"Month {t}: Produce {value(model_q2.P[t])} units, Subcontract {value(model_q2.S[t])} units, Inventory {value(model_q2.I[t])} units")

QUESTION 2
Optimal Profit: 3470700.0
Production Plan:
Month 1: Produce 160.0 units, Subcontract 70.0 units, Inventory 100.0 units
Month 2: Produce 160.0 units, Subcontract 150.0 units, Inventory 100.0 units
Month 3: Produce 160.0 units, Subcontract 210.0 units, Inventory 100.0 units
Month 4: Produce 160.0 units, Subcontract 450.0 units, Inventory 100.0 units
Month 5: Produce 160.0 units, Subcontract 220.0 units, Inventory 100.0 units
Month 6: Produce 160.0 units, Subcontract 140.0 units, Inventory 100.0 units
Month 7: Produce 160.0 units, Subcontract 230.0 units, Inventory 100.0 units
Month 8: Produce 160.0 units, Subcontract 60.0 units, Inventory 100.0 units
Month 9: Produce 160.0 units, Subcontract 180.0 units, Inventory 100.0 units
Month 10: Produce 160.0 units, Subcontract 235.0 units, Inventory 100.0 units
Month 11: Produce 160.0 units, Subcontract 242.0 units, Inventory 100.0 units
Month 12: Produce 160.0 units, Subcontract 387.0 units, Inventory 250.0 units


## Part 3 — Promotion Timing

A price drop from $2,800 to $2,520/ton is applied to one chosen month. The promotion increases same-month demand by 40% and pulls 20% forward from each of the two following months (forward buying).

### April Promotion

In [8]:
model_q3 = build_profit_model(demand_april, price_april)
solver = SolverFactory('glpk')
result_q3 = solver.solve(model_q3)

print("Optimal Profit:", value(model_q3.profit))

print("Production Plan:")
for t in model_q3.months:
    print(f"Month {t}: Produce {value(model_q3.P[t])} units, Subcontract {value(model_q3.S[t])} units, Inventory {value(model_q3.I[t])} units")

QUESTION 3 - Promotion in April
Optimal Profit: 3571700.0
Production Plan:
Month 1: Produce 160.0 units, Subcontract 70.0 units, Inventory 100.0 units
Month 2: Produce 160.0 units, Subcontract 150.0 units, Inventory 100.0 units
Month 3: Produce 160.0 units, Subcontract 210.0 units, Inventory 100.0 units
Month 4: Produce 160.0 units, Subcontract 830.0 units, Inventory 100.0 units
Month 5: Produce 160.0 units, Subcontract 144.0 units, Inventory 100.0 units
Month 6: Produce 160.0 units, Subcontract 80.0 units, Inventory 100.0 units
Month 7: Produce 160.0 units, Subcontract 230.0 units, Inventory 100.0 units
Month 8: Produce 160.0 units, Subcontract 60.0 units, Inventory 100.0 units
Month 9: Produce 160.0 units, Subcontract 180.0 units, Inventory 100.0 units
Month 10: Produce 160.0 units, Subcontract 235.0 units, Inventory 100.0 units
Month 11: Produce 160.0 units, Subcontract 242.0 units, Inventory 100.0 units
Month 12: Produce 160.0 units, Subcontract 387.0 units, Inventory 250.0 units


### August Promotion

In [9]:
model_q4 = build_profit_model(demand_august, price_august)
solver = SolverFactory('glpk')
result_q4 = solver.solve(model_q4)

print("Optimal Profit:", value(model_q4.profit))

print("Production Plan:")
for t in model_q4.months:
    print(f"Month {t}: Produce {value(model_q4.P[t])} units, Subcontract {value(model_q4.S[t])} units, Inventory {value(model_q4.I[t])} units")

QUESTION 4 - Promotion in August
Optimal Profit: 3479700.0
Production Plan:
Month 1: Produce 160.0 units, Subcontract 70.0 units, Inventory 100.0 units
Month 2: Produce 160.0 units, Subcontract 150.0 units, Inventory 100.0 units
Month 3: Produce 160.0 units, Subcontract 210.0 units, Inventory 100.0 units
Month 4: Produce 160.0 units, Subcontract 450.0 units, Inventory 100.0 units
Month 5: Produce 160.0 units, Subcontract 220.0 units, Inventory 100.0 units
Month 6: Produce 160.0 units, Subcontract 140.0 units, Inventory 100.0 units
Month 7: Produce 160.0 units, Subcontract 230.0 units, Inventory 100.0 units
Month 8: Produce 160.0 units, Subcontract 295.0 units, Inventory 100.0 units
Month 9: Produce 160.0 units, Subcontract 112.0 units, Inventory 100.0 units
Month 10: Produce 160.0 units, Subcontract 156.0 units, Inventory 100.0 units
Month 11: Produce 160.0 units, Subcontract 242.0 units, Inventory 100.0 units
Month 12: Produce 160.0 units, Subcontract 387.0 units, Inventory 250.0 unit

## Part 4 — Competitive Scenarios (Q&H vs. Unilock)

The market is a duopoly. When one firm promotes and the other does not, the promoter gains a 40% consumption uplift while the non-promoter loses 50% of that month's demand. When both promote in the same month, neither gains consumption share — both only see the forward buying effect (25% pulled from each of the two following months).

### Scenario A — Unilock promotes April, Q&H does not

In [10]:
demand_q5_qh = base_demand.copy()
price_q5_qh = base_price.copy()

demand_q5_qh[4] = 0.5 * base_demand[4]   # April demand drops by 50%

demand_q5_unilock = demand_april
price_q5_unilock = price_april


model_q5_qh = build_profit_model(demand_q5_qh, price_q5_qh)
model_q5_unilock = build_profit_model(demand_q5_unilock, price_q5_unilock)

solver = SolverFactory('glpk')
result_q5_qh = solver.solve(model_q5_qh)
result_q5_unilock = solver.solve(model_q5_unilock)


print("Q&H Profit:", value(model_q5_qh.profit))
print("Unilock Profit:", value(model_q5_unilock.profit))





QUESTION 5
Q&H Profit: 2997950.0
Unilock Profit: 3571700.0


### Scenario B — Q&H promotes April, Unilock does not

In [11]:

demand_q6_qh = demand_april.copy()
price_q6_qh = price_april.copy()

demand_q6_unilock = base_demand.copy()
price_q6_unilock = base_price.copy()

demand_q6_unilock[4] = 0.5 * base_demand[4]

model_q6_qh = build_profit_model(demand_q6_qh, price_q6_qh)
model_q6_unilock = build_profit_model(demand_q6_unilock, price_q6_unilock)

solver = SolverFactory('glpk')
result_q6_qh = solver.solve(model_q6_qh)
result_q6_unilock = solver.solve(model_q6_unilock)


print("Q&H Profit:", value(model_q6_qh.profit))
print("Unilock Profit:", value(model_q6_unilock.profit))

QUESTION 6
Q&H Profit: 3571700.0
Unilock Profit: 2997950.0


### Scenario C — Q&H promotes August, Unilock does not

In [12]:

demand_q7_qh = demand_august.copy()
price_q7_qh = price_august.copy()

demand_q7_unilock = base_demand.copy()
price_q7_unilock = base_price.copy()
demand_q7_unilock[8] = 0.5 * base_demand[8]

model_q7_qh = build_profit_model(demand_q7_qh, price_q7_qh)
model_q7_unilock = build_profit_model(demand_q7_unilock, price_q7_unilock)

solver = SolverFactory('glpk')
result_q7_qh = solver.solve(model_q7_qh)
result_q7_unilock = solver.solve(model_q7_unilock)

print("Q&H Profit:", value(model_q7_qh.profit))
print("Unilock Profit:", value(model_q7_unilock.profit))

QUESTION 7
Q&H Profit: 3479700.0
Unilock Profit: 3295200.0


### Scenario D — Both promote in April

In [13]:

demand_both_april = base_demand.copy()
price_both_april = base_price.copy()

price_both_april[4] = 2520
demand_both_april[4] = 780
demand_both_april[5] = 285
demand_both_april[6] = 225

model_q8_qh = build_profit_model(demand_both_april, price_both_april)
model_q8_unilock = build_profit_model(demand_both_april, price_both_april)

solver = SolverFactory('glpk')
result_q8_qh = solver.solve(model_q8_qh)
result_q8_unilock = solver.solve(model_q8_unilock)

print("Q&H Profit:", value(model_q8_qh.profit))
print("Unilock Profit:", value(model_q8_unilock.profit))

QUESTION 8
Q&H Profit: 3252300.0
Unilock Profit: 3252300.0


### Scenario E — Both promote in August

In [14]:

demand_both_august = base_demand.copy()
price_both_august = base_price.copy()

price_both_august[8] = 2520
demand_both_august[8] = 220 + 0.25 * 340 + 0.25 * 395
demand_both_august[9] = 0.75 * 340
demand_both_august[10] = 0.75 * 395

model_q9_qh = build_profit_model(demand_both_august, price_both_august)
model_q9_unilock = build_profit_model(demand_both_august, price_both_august)

solver = SolverFactory('glpk')
result_q9_qh = solver.solve(model_q9_qh)
result_q9_unilock = solver.solve(model_q9_unilock)

print("Q&H Profit:", value(model_q9_qh.profit))
print("Unilock Profit:", value(model_q9_unilock.profit))

QUESTION 9
Q&H Profit: 3357650.0
Unilock Profit: 3357650.0


### Scenario F — Q&H promotes April, Unilock promotes August

In [15]:
demand_q10_qh = base_demand.copy()
price_q10_qh = base_price.copy()

price_q10_qh[4] = 2520
demand_q10_qh[4] = 990
demand_q10_qh[5] = 304
demand_q10_qh[6] = 240
demand_q10_qh[8] = 110

demand_q10_unilock = base_demand.copy()
price_q10_unilock = base_price.copy()

demand_q10_unilock[4] = 305
price_q10_unilock[8] = 2520
demand_q10_unilock[8] = 455
demand_q10_unilock[9] = 272
demand_q10_unilock[10] = 316

model_q10_qh = build_profit_model(demand_q10_qh, price_q10_qh)
model_q10_unilock = build_profit_model(demand_q10_unilock, price_q10_unilock)

solver = SolverFactory('glpk')
result_q10_qh = solver.solve(model_q10_qh)
result_q10_unilock = solver.solve(model_q10_unilock)

print("Q&H Profit:", value(model_q10_qh.profit))
print("Unilock Profit:", value(model_q10_unilock.profit))

QUESTION 10
Q&H Profit: 3396200.0
Unilock Profit: 3006950.0


### Scenario G — Q&H promotes August, Unilock promotes April

In [16]:
demand_q11_qh = base_demand.copy()
price_q11_qh = base_price.copy()

demand_q11_qh[4] = 305
price_q11_qh[8] = 2520
demand_q11_qh[8] = 455
demand_q11_qh[9] = 272
demand_q11_qh[10] = 316

demand_q11_unilock = base_demand.copy()
price_q11_unilock = base_price.copy()

price_q11_unilock[4] = 2520
demand_q11_unilock[4] = 990
demand_q11_unilock[5] = 304
demand_q11_unilock[6] = 240
demand_q11_unilock[8] = 110

model_q11_qh = build_profit_model(demand_q11_qh, price_q11_qh)
model_q11_unilock = build_profit_model(demand_q11_unilock, price_q11_unilock)

solver = SolverFactory('glpk')
result_q11_qh = solver.solve(model_q11_qh)
result_q11_unilock = solver.solve(model_q11_unilock)

print("Q&H Profit:", value(model_q11_qh.profit))
print("Unilock Profit:", value(model_q11_unilock.profit))

QUESTION 11
Q&H Profit: 3006950.0
Unilock Profit: 3396200.0


## Part 5 — Strategic Analysis

### Payoff Summary

| Q&H \ Unilock | No promotion | Promotes April | Promotes August |
|---|---|---|---|
| **No promotion** | (3,470,700 / 3,470,700) | (2,997,950 / 3,571,700) | (3,295,200 / 3,479,700) |
| **Promotes April** | (3,571,700 / 2,997,950) | (3,252,300 / 3,252,300) | (3,396,200 / 3,006,950) |
| **Promotes August** | (3,479,700 / 3,295,200) | (3,006,950 / 3,396,200) | (3,357,650 / 3,357,650) |

### Maximin Strategy

Q&H's worst-case profit by action:
- No promotion → $2,997,950 (if Unilock promotes April)
- Promote April → $3,252,300 (if both promote April)  
- Promote August → $3,006,950 (if Unilock promotes April)

**Maximin decision: promote in April.** It guarantees the highest profit floor ($3,252,300) regardless of Unilock's move.

### Coordination

If both firms coordinated, the optimal split is one promotes April and the other August — avoiding same-month demand cancellation. This is mathematically superior but constitutes illegal collusion (price-fixing / market allocation) under antitrust law and is not a viable business strategy.
